### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots “talk” because they’re natural **vocal learners** who use sound to communicate—and they’re clever enough to copy the sounds they hear. Here’s the science (and a little fun trivia) behind it:\n\n| What’s happening | Why it matters |\n|------------------|----------------|\n| **Vocal learning** | Parrots have a highly developed auditory cortex and a sophisticated vocal organ called a **syrinx** (the bird equivalent of a human larynx). This lets them form new sounds on the fly. |\n| **Social bonding** | In the wild, parrots use their voices to keep flock members close, signal food, warn of predators, and maintain social hierarchies. Mimicking a human voice can be a way to “talk” to the most important member of their social group—us! |\n| **Attention‑seeking** | A parrot that can mimic a phrase or word often gets more food, attention, or affection. The reward system in their brains reinforces the behavior. |\n| **Imitation as a learning tool** | By copying sounds,

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'We need to call the get_weather function.', 'tool_calls': [{'id': 'fc_56672952-9dbe-473c-bc5e-f90c50be20da', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 127, 'total_tokens': 160, 'completion_time': 0.038333668, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.006227085, 'prompt_tokens_details': None, 'queue_time': 0.393939941, 'total_time': 0.044560753}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d23c14756c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a116ff-cb6f-7652-899b-89c2efccee25-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_56672952-9dbe-473c-bc5e-f90c50be20da', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 127, 'output_t

### Tool Execution Loops

In [11]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)
print(final_response.content)
# "The current weather in Boston is 72°F and sunny."

content='The weather in Boston is sunny.' additional_kwargs={'reasoning_content': 'User asked: "What\'s the weather in Boston?" We answered: "It\'s sunny in Boston". That is fine. We may want to add some details like temperature, humidity, etc. But we only have the function output. So we should respond with the weather. Possibly mention we used the function. The function gave "It\'s sunny in Boston". That\'s fine.'} response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 155, 'total_tokens': 244, 'completion_time': 0.110359249, 'completion_tokens_details': {'reasoning_tokens': 73}, 'prompt_time': 0.008785324, 'prompt_tokens_details': None, 'queue_time': 0.346491611, 'total_time': 0.119144573}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a11714-7a45-7d01-a8c8-af82e8de2aa9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata=

In [12]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_db130b81-bd8c-4660-9686-94c6a4c8203a', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 126, 'total_tokens': 162, 'completion_time': 0.039802606, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.006120192, 'prompt_tokens_details': None, 'queue_time': 0.346999613, 'total_time': 0.045922798}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a11714-7818-78d0-8a6b-0e2957c3332e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_db130b81-bd8c-4660-9686-94c6a4c8